# SnowHealth Demand Forecasting - Model Training
Train two ML models (XGBoost and LightGBM) with Snowflake Experiment Tracking.
Log parameters, metrics, feature importance artifacts, and models to the registry.

In [ ]:
# Initialize
import sys, os
work_dir = os.getcwd()
sys.path.insert(0, os.path.join(work_dir, 'source'))

from snowflake.snowpark.context import get_active_session
import config as cfg
from model_utils import compute_metrics, get_feature_importance, save_feature_importance_artifact, compare_models
from feature_engineering import get_feature_columns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_MODELS_SCHEMA}").collect()
print(f"Session ready: {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}")

In [ ]:
# Load engineered features
df = session.table(f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES").to_pandas()
print(f"Loaded {len(df):,} rows")

# Identify feature columns (all numeric, excluding target and IDs)
feature_cols = get_feature_columns(df, cfg.TARGET_COLUMN)
print(f"Feature columns ({len(feature_cols)}): {feature_cols[:10]}...")

# Prepare train/test split
X = df[feature_cols].astype(float)
y = df[cfg.TARGET_COLUMN].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Initialize Experiment Tracking
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(
    session=session,
    database_name=cfg.SNOWHEALTH_DATABASE,
    schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA
)
exp.set_experiment(cfg.SNOWHEALTH_EXPERIMENT_NAME)
print(f"Experiment set: {cfg.SNOWHEALTH_EXPERIMENT_NAME}")

In [ ]:
# ===== MODEL 1: XGBoost =====
from xgboost import XGBRegressor

xgb_model = XGBRegressor(**cfg.XGBOOST_BASELINE_PARAMS)

with exp.start_run("xgboost_baseline"):
    # Log parameters
    exp.log_params(cfg.XGBOOST_BASELINE_PARAMS)
    exp.log_param("model_type", "XGBRegressor")
    exp.log_param("n_features", len(feature_cols))
    exp.log_param("train_size", len(X_train))
    
    # Train
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    # Evaluate
    xgb_preds = xgb_model.predict(X_test)
    xgb_metrics = compute_metrics(y_test.values, xgb_preds)
    exp.log_metrics(xgb_metrics)
    
    # Feature importance artifact
    xgb_fi = get_feature_importance(xgb_model, feature_cols, "xgboost")
    xgb_fi_path = save_feature_importance_artifact(xgb_fi, "xgboost_baseline")
    exp.log_artifact(xgb_fi_path, artifact_path="feature_importance")
    
    print("XGBoost Baseline Results:")
    for k, v in xgb_metrics.items():
        print(f"  {k}: {v}")

In [ ]:
# ===== MODEL 2: LightGBM =====
from lightgbm import LGBMRegressor

lgbm_model = LGBMRegressor(**cfg.LIGHTGBM_BASELINE_PARAMS)

with exp.start_run("lightgbm_baseline"):
    # Log parameters
    exp.log_params(cfg.LIGHTGBM_BASELINE_PARAMS)
    exp.log_param("model_type", "LGBMRegressor")
    exp.log_param("n_features", len(feature_cols))
    exp.log_param("train_size", len(X_train))
    
    # Train
    lgbm_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
    )
    
    # Evaluate
    lgbm_preds = lgbm_model.predict(X_test)
    lgbm_metrics = compute_metrics(y_test.values, lgbm_preds)
    exp.log_metrics(lgbm_metrics)
    
    # Feature importance artifact
    lgbm_fi = get_feature_importance(lgbm_model, feature_cols, "lightgbm")
    lgbm_fi_path = save_feature_importance_artifact(lgbm_fi, "lightgbm_baseline")
    exp.log_artifact(lgbm_fi_path, artifact_path="feature_importance")
    
    print("LightGBM Baseline Results:")
    for k, v in lgbm_metrics.items():
        print(f"  {k}: {v}")

In [ ]:
# Compare models side-by-side
results = {
    "XGBoost Baseline": xgb_metrics,
    "LightGBM Baseline": lgbm_metrics,
}
comparison_df = compare_models(results)
print("\n" + "=" * 60)
print("MODEL COMPARISON (sorted by RMSE)")
print("=" * 60)
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize feature importance for both models
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# XGBoost top 15
xgb_top = xgb_fi.head(15)
axes[0].barh(range(len(xgb_top)), xgb_top['Importance_Pct'].values, color='#cc0000')
axes[0].set_yticks(range(len(xgb_top)))
axes[0].set_yticklabels(xgb_top['Feature'].values)
axes[0].set_xlabel('Importance (%)')
axes[0].set_title('XGBoost - Top 15 Features', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()

# LightGBM top 15
lgbm_top = lgbm_fi.head(15)
axes[1].barh(range(len(lgbm_top)), lgbm_top['Importance_Pct'].values, color='#333333')
axes[1].set_yticks(range(len(lgbm_top)))
axes[1].set_yticklabels(lgbm_top['Feature'].values)
axes[1].set_xlabel('Importance (%)')
axes[1].set_title('LightGBM - Top 15 Features', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.suptitle('Feature Importance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Register the best model to Snowflake Model Registry (WAREHOUSE target)
from snowflake.ml.registry import Registry
from snowflake.ml.model.model_signature import infer_signature

registry = Registry(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)

# Pick the best model by RMSE
if lgbm_metrics["rmse"] <= xgb_metrics["rmse"]:
    best_model, best_model_name, best_metrics = lgbm_model, "LightGBM", lgbm_metrics
else:
    best_model, best_model_name, best_metrics = xgb_model, "XGBoost", xgb_metrics
sig = infer_signature(X_train.head(10), y_train.head(10))

session.sql("DROP MODEL IF EXISTS SNOWHEALTH_ML_DB.SNOWHEALTH_MODELS.SNOWHEALTH_DEMAND_FORECAST").collect()
print("Dropped existing model")

mv = registry.log_model(
    best_model,
    model_name=cfg.SNOWHEALTH_MODEL_NAME,
    version_name="baseline_v1",
    signatures={"predict": sig},
    comment=f"SNOWHEALTH demand forecast baseline - {best_model_name}",
    metrics=best_metrics,
    sample_input_data=X_train.head(10),
    target_platforms=["WAREHOUSE"],
)
print(f"Model registered with WAREHOUSE target: {cfg.SNOWHEALTH_MODEL_NAME} v baseline_v1 ({best_model_name})")

In [ ]:
# Verify registration and test prediction
model_ref = registry.get_model(cfg.SNOWHEALTH_MODEL_NAME)
versions = model_ref.versions()
print(f"Model: {cfg.SNOWHEALTH_MODEL_NAME}")
for v in versions:
    print(f"  Version: {v.version_name}, Comment: {v.comment}")

# Test warehouse-based prediction
test_pred = mv.run(X_test.head(5))
print(f"\nSample predictions:")
print(test_pred)

print("\n" + "=" * 60)
print("MODEL TRAINING COMPLETE")
print("=" * 60)
print(f"Experiment: {cfg.SNOWHEALTH_EXPERIMENT_NAME}")
print(f"Models trained: XGBoost, LightGBM")
print(f"Best model: {best_model_name}")
print(f"Registered: {cfg.SNOWHEALTH_MODEL_NAME} v baseline_v1")
print("=" * 60)
print("Proceed to 04_hyperparameter_tuning.ipynb")